# 02 — Fine-tune ByT5 for Akkadian OCR Correction

Reads pre-generated pairs from `results/` and fine-tunes `google/byt5-small`.

**What you need in `results/` before running this:**
- `synthetic_pairs.jsonl` — generated locally by `scripts/generate_synthetic_pairs.py` ✅
- `ocr_pairs.jsonl` — generated by `notebooks/01_boxes_ocr.ipynb` ✅

**Output:** fine-tuned model saved to `results/byt5-akkadian/`

### How to run on Colab
1. Upload the whole `byt5-akkadian-ocr/` folder to Google Drive
2. Open this notebook in Colab, connect to a **T4 GPU** runtime
3. Set `DRIVE_REPO_PATH` below to match your Drive path
4. Run all cells — the model checkpoint saves back to Drive automatically

In [ ]:
# ── Colab / local setup ───────────────────────────────────────────────────────
import os, sys

ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # ← Change this to wherever you uploaded the repo folder in Drive
    DRIVE_REPO_PATH = '/content/drive/MyDrive/byt5-akkadian-ocr'

    sys.path.insert(0, DRIVE_REPO_PATH)
    os.chdir(DRIVE_REPO_PATH)

    !pip install -q transformers datasets sacrebleu python-Levenshtein accelerate
else:
    sys.path.insert(0, '..')
    os.chdir('..')

print(f'Working directory: {os.getcwd()}')

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

from src.metrics.evaluate import full_report

## 1. Load pre-generated pairs

`synthetic_pairs.jsonl` was generated by `scripts/generate_synthetic_pairs.py` and already
contains the train/val/test split. We just load and filter by the `split` field.

If `ocr_pairs.jsonl` exists (from the Tesseract notebook), those pairs are added to the
training set only — they don't have split labels so we don't use them for val/test.

In [ ]:
RESULTS_DIR = Path('results')
SYNTHETIC_PAIRS = RESULTS_DIR / 'synthetic_pairs.jsonl'
OCR_PAIRS       = RESULTS_DIR / 'ocr_pairs.jsonl'

assert SYNTHETIC_PAIRS.exists(), f'Missing {SYNTHETIC_PAIRS} — run scripts/generate_synthetic_pairs.py first'

# Load synthetic pairs, split by the pre-assigned split field
train_pairs, val_pairs, test_pairs = [], [], []

with open(SYNTHETIC_PAIRS, encoding='utf-8') as f:
    for line in f:
        rec = json.loads(line)
        pair = (rec['noisy'], rec['gold'])
        if   rec['split'] == 'train': train_pairs.append(pair)
        elif rec['split'] == 'val':   val_pairs.append(pair)
        elif rec['split'] == 'test':  test_pairs.append(pair)

print(f'Synthetic — train: {len(train_pairs)}, val: {len(val_pairs)}, test: {len(test_pairs)}')

# Merge Tesseract pairs into train only (if available)
if OCR_PAIRS.exists():
    ocr = [json.loads(l) for l in OCR_PAIRS.read_text().splitlines()]
    ocr_train = [(r['noisy'], r['gold']) for r in ocr]
    train_pairs.extend(ocr_train)
    print(f'Added {len(ocr_train)} Tesseract pairs → train total: {len(train_pairs)}')
else:
    print('No ocr_pairs.jsonl found — using synthetic pairs only')

# Save test pairs separately for the evaluation notebook
with open(RESULTS_DIR / 'test_pairs.jsonl', 'w', encoding='utf-8') as f:
    for noisy, gold in test_pairs:
        f.write(json.dumps({'noisy': noisy, 'gold': gold}, ensure_ascii=False) + '\n')
print(f'Saved test_pairs.jsonl ({len(test_pairs)} pairs)')

## 2. Tokenize

In [ ]:
MODEL_NAME = 'google/byt5-small'
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_dataset(pairs):
    noisy_list = [p[0] for p in pairs]
    gold_list  = [p[1] for p in pairs]
    enc = tokenizer(
        noisy_list, text_target=gold_list,
        max_length=MAX_LEN, truncation=True, padding=False,
    )
    return Dataset.from_dict(enc)

ds = DatasetDict({
    'train':      make_dataset(train_pairs),
    'validation': make_dataset(val_pairs),
    'test':       make_dataset(test_pairs),
})
print(ds)

## 3. Train

On a free Colab T4 GPU (~15GB VRAM), 5 epochs over ~59k training pairs takes roughly 2-4 hours.
The best checkpoint (lowest validation loss) is loaded automatically at the end.

In [ ]:
CHECKPOINT_DIR = str(RESULTS_DIR / 'byt5-akkadian')

model    = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    fp16=True,
    predict_with_generate=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=50,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    tokenizer=tokenizer,
    data_collator=collator,
)

trainer.train()

## 4. Run predictions on the test set

In [ ]:
preds_output = trainer.predict(ds['test'])
pred_ids     = preds_output.predictions
pred_ids     = np.where(pred_ids != -100, pred_ids, tokenizer.pad_token_id)
decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

with open(RESULTS_DIR / 'model_predictions.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(decoded_preds))

print(f'Saved {len(decoded_preds)} predictions to results/model_predictions.txt')

## 5. Evaluate — model vs baseline

The baseline is the raw noisy input scored against gold — i.e., "do nothing".
The model must beat the baseline to have learned anything useful.

In [ ]:
test_noisy = [p[0] for p in test_pairs]
test_gold  = [p[1] for p in test_pairs]

baseline = full_report(test_noisy,    test_gold, label='baseline (noisy input, no model)')
model_r  = full_report(decoded_preds, test_gold, label='byt5-small fine-tuned')

for report in [baseline, model_r]:
    print(f"\n=== {report['label']} ===")
    print(f"  Exact match : {report['exact_match']:.3f}")
    print(f"  CER         : {report['cer']:.3f}  (lower is better)")
    print(f"  chrF++      : {report['chrf']:.2f}")
    print(f"  BLEU        : {report['bleu']:.2f}")

# Show a few example corrections
print('\n── Sample corrections (10 random from test) ──')
indices = random.sample(range(len(test_gold)), min(10, len(test_gold)))
for i in indices:
    print(f'  noisy : {test_noisy[i]}')
    print(f'  model : {decoded_preds[i]}')
    print(f'  gold  : {test_gold[i]}')
    print()